<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES_Stage6C_Cell_6C_4K0_Final_Integrated_Package_Freeze_FINAL_CORRECTED_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GES Stage 6C — Cell 6C-4K0
## Final Integrated Stage 6C Package Freeze — Final Corrected V2

This version uses the already-defined `EXPECTED_STAGE6B_ROWS` constant for reliability-table reconciliation.


In [1]:
# ==================================================================================================
# COLAB NOTEBOOK FILE NAME:
# GES_Stage6C_Cell_6C_4K0_Final_Integrated_Package_Freeze.ipynb
#
# STAGE 6C STEP 4K — CELL 6C-4K0
# FINAL INTEGRATED STAGE 6C PACKAGE FREEZE, SEMANTIC READBACK, AND FRESH REVERIFICATION
#
# Scientific and administrative boundary:
#   - This cell does not calculate a new scientific result.
#   - This cell does not alter any score, outcome, model, threshold, linkage decision, row order,
#     gene assignment, censoring rule, cohort membership, or prior frozen artifact.
#   - This cell verifies and integrates the complete Stage 6C evidence package after all eight
#     preflight-identified result categories have been independently materialized.
#   - Experiment 2 is not started or authorized by this cell. A separate explicit go/no-go decision
#     is required after this final freeze.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import platform
import re
import sys
import time

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. PROJECT ROOT, IMMUTABLE INPUTS, AND FINAL OUTPUT LOCATIONS
# --------------------------------------------------------------------------------------------------

NOTEBOOK_FILENAME = "GES_Stage6C_Cell_6C_4K0_Final_Integrated_Package_Freeze.ipynb"
ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")
if not ROOT.exists():
    raise FileNotFoundError(f"Project directory does not exist: {ROOT}")

STAGE6_DIR = ROOT / "data_processed/stage6_temporal_validation"
CONFIG_ROOT = ROOT / "configs/stage6_temporal_validation"

STAGE6B = STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
STAGE6B_SHA256 = "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
EXPECTED_STAGE6B_ROWS = 66_636
EXPECTED_STAGE6B_COLUMNS = 79
EXPECTED_STAGE6B_EVENTS = 6_485
EXPECTED_STAGE6B_NEGATIVES = 60_151

NESTED_DIR = STAGE6_DIR / "nested_scv_secondary_outcomes"
NESTED_PACKAGE = NESTED_DIR / "stage6c_nested_scv_secondary_outcomes_record_level_v1.parquet"
NESTED_PACKAGE_SHA256 = "18b3d75b62e8d1b891dcd88eb951b4aa767aa6c88570369004bd08b9c3de7fc2"
NESTED_MANIFEST = NESTED_DIR / "stage6c_nested_scv_secondary_outcomes_manifest_v1.json"
NESTED_MANIFEST_SHA256 = "b89626648af773b79053515f81dc1b7afee1ca3e516510a2bf52aa5110c5edc5"
EXPECTED_NESTED_ROWS = 66_636
EXPECTED_NESTED_COLUMNS = 28

FIGURE_TABLE_DIR = ROOT / (
    "outputs/tables/stage6_temporal_validation/"
    "stage6c_4a0_final_figures_error_review_v1"
)
FIGURE_DIR = ROOT / (
    "outputs/figures/stage6_temporal_validation/"
    "stage6c_4a0_final_figures_error_review_v1"
)
FIGURE_QC = ROOT / (
    "outputs/quality_checks/stage6_temporal_validation/"
    "stage6c_4a0_final_figures_error_review_v1/"
    "stage6c_4a0_figures_error_review_qc_v1.json"
)
FIGURE_QC_SHA256 = "15b2fc96851fb078db72e1e01932dd21a1c36b8abff8c66571e863024e8e55ee"
FIGURE_MANIFEST = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4a0_final_figures_error_review_v1/"
    "stage6c_4a0_figures_error_review_manifest_v1.json"
)
FIGURE_MANIFEST_SHA256 = "f11332ac6bdeaaf59813b1ee1ad0df129fae30b8438337def2f57f1c7dbbf5d7"
EXPECTED_4A_ARTIFACTS = 18
EXPECTED_4A_TABLES = 12
EXPECTED_4A_FIGURES = 6

PACKAGE_NAME = "stage6c_4k0_final_integrated_package_freeze_v1"
TABLE_DIR = ROOT / f"outputs/tables/stage6_temporal_validation/{PACKAGE_NAME}"
QC_DIR = ROOT / f"outputs/quality_checks/stage6_temporal_validation/{PACKAGE_NAME}"
MANIFEST_DIR = ROOT / f"configs/stage6_temporal_validation/{PACKAGE_NAME}"
for directory in (TABLE_DIR, QC_DIR, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

P = {
    "inventory": TABLE_DIR / "stage6c_final_integrated_artifact_inventory_v1.csv",
    "coverage": TABLE_DIR / "stage6c_final_required_category_coverage_v1.csv",
    "conclusions": TABLE_DIR / "stage6c_final_scientific_conclusion_register_v1.csv",
    "package_index": TABLE_DIR / "stage6c_final_package_index_v1.json",
    "qc": QC_DIR / "stage6c_4k0_final_integrated_package_freeze_qc_v1.json",
    "manifest": MANIFEST_DIR / "stage6c_4k0_final_integrated_package_manifest_v1.json",
}

if P["manifest"].exists():
    CREATED_UTC = json.loads(P["manifest"].read_text(encoding="utf-8"))["created_utc"]
elif P["qc"].exists():
    CREATED_UTC = json.loads(P["qc"].read_text(encoding="utf-8"))["created_utc"]
else:
    CREATED_UTC = datetime.now(timezone.utc).isoformat()


# --------------------------------------------------------------------------------------------------
# 2. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    path = Path(path)
    return path.with_name(path.name + ".sha256")


def read_sidecar_hash(path: Path) -> str:
    text = Path(path).read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def sidecar_ok(path: Path) -> bool:
    path = Path(path)
    output = sidecar_path(path)
    return output.exists() and read_sidecar_hash(output) == sha(path)


def native(value):
    if isinstance(value, dict):
        return {str(key): native(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [native(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return native(value.tolist())
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is pd.NA:
        return None
    return value


def stable_write_bytes(path: Path, payload: bytes) -> str:
    """Create atomically; on rerun accept only byte-identical content."""
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    temporary.write_bytes(payload)
    new_hash = sha(temporary)
    if path.exists():
        if sha(path) != new_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Refusing to overwrite nonidentical frozen artifact: {path}")
        temporary.unlink(missing_ok=True)
    else:
        os.replace(temporary, path)
    return sha(path)


def write_csv(path: Path, frame: pd.DataFrame) -> str:
    payload = frame.to_csv(
        index=False,
        lineterminator="\n",
        float_format="%.12g",
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_json(path: Path, obj) -> str:
    payload = (
        json.dumps(
            native(obj),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_sidecar(path: Path) -> Path:
    output = sidecar_path(path)
    stable_write_bytes(output, f"{sha(path)}  {Path(path).name}\n".encode("utf-8"))
    return output


def normalized_token(text: str) -> str:
    return re.sub(r"[^A-Z0-9]+", "_", str(text).upper()).strip("_")


def path_from_entry(entry: dict) -> Path:
    raw = (
        entry.get("path")
        or entry.get("artifact_path")
        or entry.get("relative_path")
        or entry.get("relative_to_project")
    )
    if raw is None:
        raise KeyError(f"Manifest artifact entry does not contain a path: {entry}")
    path = Path(str(raw))
    return path if path.is_absolute() else ROOT / path


def recursively_contains_true_experiment2_flag(obj) -> bool:
    if isinstance(obj, dict):
        for key, value in obj.items():
            key_token = normalized_token(key)
            if "EXPERIMENT_2_STARTED" in key_token and value is True:
                return True
            if recursively_contains_true_experiment2_flag(value):
                return True
    elif isinstance(obj, list):
        return any(recursively_contains_true_experiment2_flag(item) for item in obj)
    return False


def semantic_shape(path: Path) -> dict:
    path = Path(path)
    suffix = path.suffix.lower()
    result = {"rows": None, "columns": None, "readback_type": suffix}
    if suffix in {".csv", ".tsv"}:
        separator = "\t" if suffix == ".tsv" else ","
        frame = pd.read_csv(path, sep=separator)
        result.update(rows=int(len(frame)), columns=int(frame.shape[1]))
    elif suffix == ".parquet":
        metadata = pq.ParquetFile(path).metadata
        result.update(rows=int(metadata.num_rows), columns=int(metadata.num_columns))
    elif suffix == ".json":
        json.loads(path.read_text(encoding="utf-8"))
        result["json_readback"] = True
    elif suffix in {".png", ".pdf", ".svg"}:
        result["bytes"] = int(path.stat().st_size)
    else:
        result["bytes"] = int(path.stat().st_size)
    return result


def verify_fixed(path: Path, expected_hash: str, label: str) -> dict:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required artifact for {label}: {path}")
    if not sidecar_ok(path):
        raise RuntimeError(f"Missing or invalid sidecar for {label}: {path}")
    observed = sha(path)
    if observed != expected_hash:
        raise RuntimeError(
            f"Hash mismatch for {label}. Expected {expected_hash}; observed {observed}."
        )
    return {
        "label": label,
        "path": str(path),
        "relative_path": str(path.relative_to(ROOT)),
        "sha256": observed,
        "bytes": int(path.stat().st_size),
        "sidecar_path": str(sidecar_path(path)),
        "sidecar_sha256": sha(sidecar_path(path)),
        "sidecar_verified": True,
        **semantic_shape(path),
    }


def locate_manifest_by_hash(expected_hash: str, cell_id: str) -> Path:
    likely = sorted(CONFIG_ROOT.rglob(f"*{cell_id.lower().replace('-', '_')}*.json"))
    all_candidates = likely + [
        path for path in sorted(CONFIG_ROOT.rglob("*.json")) if path not in set(likely)
    ]
    matches = []
    for candidate in all_candidates:
        try:
            if sha(candidate) == expected_hash:
                matches.append(candidate)
        except OSError:
            continue
    unique = sorted(set(matches))
    if len(unique) != 1:
        raise RuntimeError(
            f"Expected exactly one manifest for {cell_id} with hash {expected_hash}; "
            f"found {len(unique)}: {unique}"
        )
    return unique[0]


def artifact_text(entry: dict, path: Path) -> str:
    values = [
        path.name,
        str(path),
        str(entry.get("artifact_key", "")),
        str(entry.get("label", "")),
        str(entry.get("description", "")),
        str(entry.get("artifact_type", "")),
    ]
    return normalized_token(" ".join(values))


def pattern_group_present(texts: list[str], alternatives: list[str]) -> bool:
    return any(
        any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in alternatives)
        for text in texts
    )


def artifact_record(
    path: Path,
    source_group: str,
    category: str,
    expected_hash: str | None = None,
    entry_label: str = "",
) -> dict:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    observed = sha(path)
    if expected_hash is not None and observed != expected_hash:
        raise RuntimeError(
            f"Artifact hash mismatch for {path}. Expected {expected_hash}; observed {observed}."
        )
    if not sidecar_ok(path):
        raise RuntimeError(f"Artifact sidecar verification failed: {path}")
    return {
        "source_group": source_group,
        "required_category": category,
        "entry_label": entry_label,
        "path": str(path),
        "relative_path": str(path.relative_to(ROOT)),
        "file_name": path.name,
        "suffix": path.suffix.lower(),
        "bytes": int(path.stat().st_size),
        "sha256": observed,
        "sidecar_path": str(sidecar_path(path)),
        "sidecar_sha256": sha(sidecar_path(path)),
        "sidecar_verified": True,
        **semantic_shape(path),
    }


# --------------------------------------------------------------------------------------------------
# 3. VERIFY IMMUTABLE STAGE 6 SOURCES
# --------------------------------------------------------------------------------------------------

source_records = []
source_records.append(verify_fixed(STAGE6B, STAGE6B_SHA256, "Stage 6B evaluable cohort"))
source_records.append(
    verify_fixed(NESTED_PACKAGE, NESTED_PACKAGE_SHA256, "Nested-SCV record-level package")
)
source_records.append(
    verify_fixed(NESTED_MANIFEST, NESTED_MANIFEST_SHA256, "Nested-SCV source manifest")
)
source_records.append(verify_fixed(FIGURE_QC, FIGURE_QC_SHA256, "Cell 6C-4A0 QC"))
source_records.append(
    verify_fixed(FIGURE_MANIFEST, FIGURE_MANIFEST_SHA256, "Cell 6C-4A0 manifest")
)

stage6b_metadata = pq.ParquetFile(STAGE6B).metadata
if (stage6b_metadata.num_rows, stage6b_metadata.num_columns) != (
    EXPECTED_STAGE6B_ROWS,
    EXPECTED_STAGE6B_COLUMNS,
):
    raise RuntimeError(
        "Stage 6B dimensions mismatch: "
        f"{stage6b_metadata.num_rows:,} × {stage6b_metadata.num_columns}"
    )

stage6b_outcome = pd.read_parquet(STAGE6B, columns=["primary_future_instability"])
stage6b_events = int(pd.to_numeric(stage6b_outcome["primary_future_instability"]).sum())
stage6b_negatives = int(len(stage6b_outcome) - stage6b_events)
if (stage6b_events, stage6b_negatives) != (
    EXPECTED_STAGE6B_EVENTS,
    EXPECTED_STAGE6B_NEGATIVES,
):
    raise RuntimeError(
        f"Stage 6B event accounting mismatch: {stage6b_events:,}/{stage6b_negatives:,}"
    )

nested_metadata = pq.ParquetFile(NESTED_PACKAGE).metadata
if (nested_metadata.num_rows, nested_metadata.num_columns) != (
    EXPECTED_NESTED_ROWS,
    EXPECTED_NESTED_COLUMNS,
):
    raise RuntimeError(
        "Nested-SCV package dimensions mismatch: "
        f"{nested_metadata.num_rows:,} × {nested_metadata.num_columns}"
    )

figure_qc_data = json.loads(FIGURE_QC.read_text(encoding="utf-8"))
figure_manifest_data = json.loads(FIGURE_MANIFEST.read_text(encoding="utf-8"))
if (figure_qc_data.get("cell") or figure_qc_data.get("cell_id")) != "6C-4A0":
    raise RuntimeError("Unexpected Cell 6C-4A0 QC identity.")
if int(figure_qc_data.get("checks_failed", 0)) != 0:
    raise RuntimeError("Cell 6C-4A0 QC contains failures.")
if (figure_manifest_data.get("cell") or figure_manifest_data.get("cell_id")) != "6C-4A0":
    raise RuntimeError("Unexpected Cell 6C-4A0 manifest identity.")
if recursively_contains_true_experiment2_flag(figure_manifest_data):
    raise RuntimeError("Cell 6C-4A0 manifest indicates Experiment 2 started.")

figure_entries = figure_manifest_data.get("artifacts", [])
if len(figure_entries) != EXPECTED_4A_ARTIFACTS:
    raise RuntimeError(
        f"Cell 6C-4A0 manifest contains {len(figure_entries)} artifacts; "
        f"expected {EXPECTED_4A_ARTIFACTS}."
    )

integrated_records = []
for record in source_records:
    integrated_records.append(
        {
            "source_group": "immutable_or_staging_source",
            "required_category": "source_lineage",
            "entry_label": record["label"],
            **{key: value for key, value in record.items() if key != "label"},
            "file_name": Path(record["path"]).name,
            "suffix": Path(record["path"]).suffix.lower(),
        }
    )

figure_artifact_records = []
for entry in figure_entries:
    path = path_from_entry(entry)
    expected_hash = entry.get("sha256") or entry.get("hash")
    if not expected_hash:
        raise RuntimeError(f"Cell 6C-4A0 artifact missing expected hash: {entry}")
    record = artifact_record(
        path,
        source_group="cell_6c_4a0",
        category="final_figures_and_error_review",
        expected_hash=str(expected_hash),
        entry_label=str(entry.get("label") or entry.get("description") or ""),
    )
    stated_sidecar_hash = entry.get("sidecar_sha256")
    if stated_sidecar_hash and sha(sidecar_path(path)) != stated_sidecar_hash:
        raise RuntimeError(f"Cell 6C-4A0 sidecar-file hash mismatch: {path}")
    figure_artifact_records.append(record)
    integrated_records.append(record)

figure_count = sum(record["suffix"] == ".png" for record in figure_artifact_records)
table_count = sum(record["suffix"] in {".csv", ".parquet"} for record in figure_artifact_records)
if (figure_count, table_count) != (EXPECTED_4A_FIGURES, EXPECTED_4A_TABLES):
    raise RuntimeError(
        f"Cell 6C-4A0 artifact composition mismatch: {figure_count} figures, {table_count} tables."
    )


# --------------------------------------------------------------------------------------------------
# 4. VERIFY ALL EIGHT INDEPENDENT RESULT-CATEGORY MANIFESTS AND THEIR ARTIFACTS
# --------------------------------------------------------------------------------------------------

CATEGORY_PACKAGES = OrderedDict(
    [
        (
            "principal_auprc_auroc_bootstrap",
            {
                "cell_id": "6C-4C0",
                "manifest_sha256": "5f260c7167ceb2b137d57119ecdf2d59f004a98cc2aab72b96435e760890e578",
                "decision_tokens": ["PASS_STAGE6C", "PRINCIPAL", "BOOTSTRAP", "MATERIALIZED", "REVERIFIED"],
                "required_artifact_groups": [
                    [r"REPLICATE", r"BOOTSTRAP"],
                    [r"INTERVAL", r"SUMMARY"],
                    [r"PAIRED"],
                    [r"QC"],
                ],
            },
        ),
        (
            "remaining_comparator_inference",
            {
                "cell_id": "6C-4D0",
                "manifest_sha256": "d7082823437de952f0e4a96204f6c439015d74e26a9f6b14f57eeffd955e2cd6",
                "decision_tokens": ["PASS_STAGE6C", "REMAINING", "COMPARATOR", "MATERIALIZED", "REVERIFIED"],
                "required_artifact_groups": [
                    [r"REPLICATE", r"BOOTSTRAP"], [r"INTERVAL"], [r"PAIRED"], [r"CONCORDANCE"], [r"QC"]
                ],
            },
        ),
        (
            "same_star_point_and_bootstrap",
            {
                "cell_id": "6C-4E0",
                "manifest_sha256": "3a91230ae06b4b776f0c3b1f4311f1a8ea52086992a1439baebc7fd7b35768a9",
                "decision_tokens": ["PASS_STAGE6C", "SAME_STAR", "MATERIALIZED", "REVERIFIED"],
                "required_artifact_groups": [
                    [r"STRATUM", r"INVENTORY"], [r"REPLICATE", r"BOOTSTRAP"], [r"INTERVAL"],
                    [r"PAIRED"], [r"HOLM", r"MULTIPLICITY"], [r"SPARSE"], [r"QC"]
                ],
            },
        ),
        (
            "gene_level_point_and_bootstrap",
            {
                "cell_id": "6C-4F0",
                "manifest_sha256": "34646aa38862bbc425a7373a90e49c336023bc8e0e37ba82c8d9df5824a2e365",
                "decision_tokens": ["PASS_STAGE6C", "GENE_LEVEL", "MATERIALIZED", "REVERIFIED"],
                "required_artifact_groups": [
                    [r"GENE.*INVENTORY", r"INVENTORY"], [r"REPLICATE", r"BOOTSTRAP"], [r"INTERVAL"],
                    [r"PAIRED"], [r"HOLM"], [r"CONCORDANCE"], [r"QC"]
                ],
            },
        ),
        (
            "exact_link_sensitivity",
            {
                "cell_id": "6C-4G0",
                "manifest_sha256": "f9586e0609f76f9adda52e153dfcb7800e77b0840a1eb076c44936777ba8611f",
                "decision_tokens": ["PASS_STAGE6C", "EXACT_LINK", "MATERIALIZED", "REVERIFIED"],
                "required_artifact_groups": [
                    [r"INVENTORY", r"ACCOUNTING"], [r"REPLICATE", r"BOOTSTRAP"], [r"INTERVAL"],
                    [r"PAIRED"], [r"HOLM"], [r"CONCORDANCE"], [r"QC"]
                ],
            },
        ),
        (
            "alternative_outcome_and_secondary_drift",
            {
                "cell_id": "6C-4H0",
                "manifest_sha256": "207eeb09c9f5d6d0f1e46058252b27c4f4906d26f918a08bb29780b7de00d2db",
                "decision_tokens": ["PASS_STAGE6C", "ALTERNATIVE_OUTCOME", "SECONDARY_DRIFT", "MATERIALIZED", "REVERIFIED"],
                "required_artifact_groups": [
                    [r"ACCOUNTING"], [r"REPLICATE", r"BOOTSTRAP"], [r"INTERVAL"], [r"PAIRED"],
                    [r"HOLM", r"MULTIPLICITY"], [r"SPARSE"], [r"CONCORDANCE"], [r"QC"]
                ],
            },
        ),
        (
            "nested_scv_record_level_and_37_record_analysis",
            {
                "cell_id": "6C-4I0",
                "manifest_sha256": "651b2fd86cfb24a6a07745cd1737821e77d1b858182afd7863a90b121b5b25ae",
                "decision_tokens": ["PASS_STAGE6C", "NESTED_SCV", "37_RECORD", "MATERIALIZED", "REVERIFIED"],
                "required_artifact_groups": [
                    [r"ACCOUNTING", r"COHORT"], [r"REPLICATE", r"BOOTSTRAP"], [r"INTERVAL"],
                    [r"PAIRED"], [r"LIMITATION"], [r"CONCORDANCE"], [r"QC"]
                ],
            },
        ),
        (
            "leave_one_gene_out_validation",
            {
                "cell_id": "6C-4J0",
                "manifest_sha256": "05c63d4a4a2ba897e73f56d91073903f069c56ff88fb648106aefa00bcf5b8cf",
                "decision_tokens": ["PASS_STAGE6C", "LEAVE_ONE_GENE_OUT", "MATERIALIZED", "REVERIFIED"],
                "required_artifact_groups": [
                    [r"TRAINING.*ACCOUNTING"], [r"HELD_OUT.*ACCOUNTING", r"HELD_OUT.*PREDICTION"],
                    [r"REPLICATE", r"BOOTSTRAP"], [r"INTERVAL"], [r"PAIRED"], [r"HOLM"],
                    [r"CONCORDANCE"], [r"QC"]
                ],
            },
        ),
    ]
)

package_verification_rows = []
category_artifact_records = {}

for category, specification in CATEGORY_PACKAGES.items():
    manifest_path = locate_manifest_by_hash(
        specification["manifest_sha256"],
        specification["cell_id"],
    )
    manifest_record = verify_fixed(
        manifest_path,
        specification["manifest_sha256"],
        f"{specification['cell_id']} category manifest",
    )
    manifest_data = json.loads(manifest_path.read_text(encoding="utf-8"))

    observed_cell = manifest_data.get("cell") or manifest_data.get("cell_id")
    if observed_cell != specification["cell_id"]:
        raise RuntimeError(
            f"Manifest identity mismatch for {category}: expected {specification['cell_id']}; "
            f"observed {observed_cell}."
        )

    decision = str(manifest_data.get("decision", ""))
    decision_normalized = normalized_token(decision)
    missing_decision_tokens = [
        token
        for token in specification["decision_tokens"]
        if normalized_token(token) not in decision_normalized
    ]
    if missing_decision_tokens:
        raise RuntimeError(
            f"Manifest decision for {category} lacks required tokens {missing_decision_tokens}: {decision}"
        )
    if recursively_contains_true_experiment2_flag(manifest_data):
        raise RuntimeError(f"Manifest for {category} indicates Experiment 2 started.")

    entries = manifest_data.get("artifacts", [])
    if not entries:
        raise RuntimeError(f"Manifest for {category} contains no artifacts.")

    records = []
    texts = []
    for entry in entries:
        path = path_from_entry(entry)
        expected_hash = entry.get("sha256") or entry.get("hash")
        if not expected_hash:
            raise RuntimeError(f"Artifact entry lacks expected SHA-256 in {manifest_path}: {entry}")
        record = artifact_record(
            path,
            source_group=specification["cell_id"],
            category=category,
            expected_hash=str(expected_hash),
            entry_label=str(
                entry.get("artifact_key")
                or entry.get("label")
                or entry.get("description")
                or ""
            ),
        )
        stated_sidecar = entry.get("sidecar_path")
        if stated_sidecar:
            stated_sidecar_path = Path(str(stated_sidecar))
            if not stated_sidecar_path.is_absolute():
                stated_sidecar_path = ROOT / stated_sidecar_path
            if stated_sidecar_path != sidecar_path(path):
                raise RuntimeError(
                    f"Manifest sidecar path differs from expected sidecar path for {path}: "
                    f"{stated_sidecar_path}"
                )
        stated_sidecar_hash = entry.get("sidecar_sha256")
        if stated_sidecar_hash and sha(sidecar_path(path)) != stated_sidecar_hash:
            raise RuntimeError(f"Manifest sidecar-file hash mismatch: {path}")
        records.append(record)
        texts.append(artifact_text(entry, path))
        integrated_records.append(record)

    for alternatives in specification["required_artifact_groups"]:
        if not pattern_group_present(texts, alternatives):
            raise RuntimeError(
                f"Semantic artifact-group verification failed for {category}; "
                f"missing one of {alternatives}. Artifact texts: {texts}"
            )

    if not any(record["suffix"] in {".csv", ".parquet"} for record in records):
        raise RuntimeError(f"No result table found in manifest for {category}.")
    if not any(record["suffix"] == ".json" and "QC" in normalized_token(record["file_name"]) for record in records):
        raise RuntimeError(f"No QC JSON found in manifest for {category}.")

    manifest_inventory_record = {
        "source_group": specification["cell_id"],
        "required_category": category,
        "entry_label": "category_manifest",
        **{key: value for key, value in manifest_record.items() if key != "label"},
        "file_name": manifest_path.name,
        "suffix": manifest_path.suffix.lower(),
    }
    integrated_records.append(manifest_inventory_record)
    category_artifact_records[category] = records

    package_verification_rows.append(
        {
            "required_category": category,
            "cell_id": specification["cell_id"],
            "manifest_path": str(manifest_path),
            "manifest_sha256": sha(manifest_path),
            "manifest_sidecar_verified": sidecar_ok(manifest_path),
            "manifest_decision": decision,
            "artifact_count": len(records),
            "all_artifacts_readable": True,
            "all_artifact_sidecars_verified": all(record["sidecar_verified"] for record in records),
            "experiment_2_started": False,
        }
    )

package_verification = pd.DataFrame(package_verification_rows)
if len(package_verification) != 8:
    raise RuntimeError("Eight independent category packages were not verified.")


# --------------------------------------------------------------------------------------------------
# 5. SEMANTIC READBACK OF THE SIX PRE-EXISTING INVENTORY-READY CATEGORIES
# --------------------------------------------------------------------------------------------------

BASE_TABLES = {
    "model_points": FIGURE_TABLE_DIR / "stage6c_model_point_estimates_for_figures_v1.csv",
    "reliability": FIGURE_TABLE_DIR / "stage6c_reliability_table_for_figure_v1.csv",
    "enrichment": FIGURE_TABLE_DIR / "stage6c_exact_rank_top_risk_enrichment_for_figure_v1.csv",
    "deciles": FIGURE_TABLE_DIR / "stage6c_exact_rank_risk_decile_event_rates_v1.csv",
    "threshold": FIGURE_TABLE_DIR / "stage6c_full_ges_frozen_threshold_error_summary_v1.csv",
    "event_components": FIGURE_TABLE_DIR / "stage6c_false_stable_event_component_summary_v1.csv",
}
for label, path in BASE_TABLES.items():
    if not path.exists() or not sidecar_ok(path):
        raise RuntimeError(f"Missing or invalid base-category evidence table {label}: {path}")

model_points = pd.read_csv(BASE_TABLES["model_points"])
required_point_columns = {"model", "point_auprc", "point_auroc"}
if len(model_points) != 9 or not required_point_columns.issubset(model_points.columns):
    raise RuntimeError("Global nine-score point-estimate table failed semantic verification.")
full_row = model_points.loc[model_points["model"].astype(str).eq("Full GES")]
if len(full_row) != 1:
    raise RuntimeError("Full GES row is absent or duplicated in global point estimates.")
if not np.isclose(float(full_row.iloc[0]["point_auprc"]), 0.112444062354, atol=5e-12):
    raise RuntimeError("Full GES AUPRC semantic value mismatch.")
if not np.isclose(float(full_row.iloc[0]["point_auroc"]), 0.535812, atol=5.1e-7):
    raise RuntimeError("Full GES AUROC semantic value mismatch.")

reliability = pd.read_csv(BASE_TABLES["reliability"])
required_reliability_columns = {
    "model_key",
    "model",
    "records",
    "mean_predicted_risk",
    "observed_event_rate",
}
if reliability.empty or not required_reliability_columns.issubset(reliability.columns):
    raise RuntimeError("Calibration/reliability table failed semantic verification.")

# Cell 6C-4A0 intentionally created the reliability figure/table for the four
# prespecified principal models, not for all nine discrimination scores.
expected_reliability_models = {
    "Full GES",
    "No-star GES",
    "Review stars",
    "Combined metadata",
}
observed_reliability_models = set(reliability["model"].astype(str))

if observed_reliability_models != expected_reliability_models:
    raise RuntimeError(
        "Reliability table principal-model coverage mismatch. "
        f"Expected {sorted(expected_reliability_models)}; "
        f"observed {sorted(observed_reliability_models)}."
    )

reliability["records"] = pd.to_numeric(
    reliability["records"],
    errors="raise",
).astype("int64")
reliability["mean_predicted_risk"] = pd.to_numeric(
    reliability["mean_predicted_risk"],
    errors="raise",
).astype(float)
reliability["observed_event_rate"] = pd.to_numeric(
    reliability["observed_event_rate"],
    errors="raise",
).astype(float)

if (reliability["records"] <= 0).any():
    raise RuntimeError("Reliability table contains a nonpositive group size.")
if not np.isfinite(
    reliability[["mean_predicted_risk", "observed_event_rate"]].to_numpy(float)
).all():
    raise RuntimeError("Reliability table contains nonfinite rate values.")
if (
    (reliability["mean_predicted_risk"] < 0.0)
    | (reliability["mean_predicted_risk"] > 1.0)
    | (reliability["observed_event_rate"] < 0.0)
    | (reliability["observed_event_rate"] > 1.0)
).any():
    raise RuntimeError("Reliability table contains values outside [0,1].")

reliability_record_totals = (
    reliability.groupby("model", sort=True)["records"].sum().astype("int64")
)
if not (
    reliability_record_totals.index.astype(str).isin(expected_reliability_models).all()
    and (reliability_record_totals == EXPECTED_STAGE6B_ROWS).all()
    and len(reliability_record_totals) == len(expected_reliability_models)
):
    raise RuntimeError(
        "Reliability groups do not reconcile to the complete 66,636-row cohort "
        "for each principal model: "
        f"{reliability_record_totals.to_dict()}"
    )

enrichment = pd.read_csv(BASE_TABLES["enrichment"])
required_enrichment_columns = {
    "selected_records",
    "selected_events",
    "selected_event_rate",
    "boundary_tie_total_records",
    "boundary_tie_selected_records",
}
if len(enrichment) != 3 or not required_enrichment_columns.issubset(enrichment.columns):
    raise RuntimeError("Enrichment/tie-audit table failed semantic verification.")
if enrichment["selected_records"].astype(int).tolist() != [3332, 6664, 13328]:
    raise RuntimeError("Exact-rank enrichment selected-record counts changed.")

deciles = pd.read_csv(BASE_TABLES["deciles"])
if deciles.empty or not any("decile" in column.lower() for column in deciles.columns):
    raise RuntimeError("Risk-decile table failed semantic verification.")

threshold = pd.read_csv(BASE_TABLES["threshold"])
if len(threshold) != 1:
    raise RuntimeError("Frozen-threshold summary must contain exactly one row.")
threshold_row = threshold.iloc[0]
expected_threshold_counts = {
    "true_positive": 442,
    "false_stable_false_negative": 6043,
    "false_unstable_false_positive": 4057,
    "true_negative": 56094,
}
for column, expected in expected_threshold_counts.items():
    if column not in threshold.columns or int(threshold_row[column]) != expected:
        raise RuntimeError(f"Frozen-threshold semantic mismatch for {column}.")
if not np.isclose(float(threshold_row["balanced_accuracy"]), 0.500355, atol=5.1e-7):
    raise RuntimeError("Frozen-threshold balanced accuracy mismatch.")
threshold_optimized_value = str(threshold_row["threshold_optimized_on_t1"]).strip().lower()
if threshold_optimized_value in {"true", "1", "1.0", "yes"}:
    raise RuntimeError("Threshold table indicates T1 optimization occurred.")

event_components = pd.read_csv(BASE_TABLES["event_components"])
if len(event_components) != 3:
    raise RuntimeError("Event-component summary must contain three frozen components.")
component_col = "event_component_column"
if component_col not in event_components.columns:
    raise RuntimeError("Event-component identifier column missing.")
new_conflict = event_components.loc[
    event_components[component_col].astype(str).eq("event_new_unresolved_conflict_at_t1")
]
if len(new_conflict) != 1:
    raise RuntimeError("New unresolved conflict component is absent or duplicated.")
if int(new_conflict.iloc[0]["component_events_in_full_cohort"]) != 4789:
    raise RuntimeError("New unresolved conflict event total changed.")
if int(new_conflict.iloc[0]["component_events_among_false_stable"]) != 4721:
    raise RuntimeError("New unresolved conflict false-stable count changed.")

base_category_evidence = {
    "global_nine_score_point_estimates": [BASE_TABLES["model_points"]],
    "calibration_and_reliability": [BASE_TABLES["reliability"]],
    "enrichment_decile_and_tie_audits": [BASE_TABLES["enrichment"], BASE_TABLES["deciles"]],
    "frozen_threshold_point_and_bootstrap": [BASE_TABLES["threshold"]],
    "event_component_analysis": [BASE_TABLES["event_components"]],
    "final_figures_and_error_review": [Path(record["path"]) for record in figure_artifact_records],
}

# Locate additional independently written threshold-bootstrap and event-component analysis evidence.
# These were already inventory-ready before Cells 6C-4C0 through 6C-4J0. The final freeze opens each
# candidate, verifies its sidecar, and requires substantive tabular content rather than relying on a name alone.

def discover_supporting_tables(path_patterns: list[str], semantic_kind: str) -> list[Path]:
    candidates = []
    for search_root in [ROOT / "outputs", ROOT / "data_processed/stage6_temporal_validation"]:
        if not search_root.exists():
            continue
        for path in search_root.rglob("*"):
            if not path.is_file() or path.name.endswith(".sha256"):
                continue
            if PACKAGE_NAME in str(path):
                continue
            if path.suffix.lower() not in {".csv", ".tsv", ".parquet", ".json"}:
                continue
            text = str(path.relative_to(ROOT)).lower().replace("\\", "/")
            if not all(re.search(pattern, text) for pattern in path_patterns):
                continue
            if not sidecar_ok(path):
                continue
            try:
                shape = semantic_shape(path)
                rows = shape.get("rows")
                if semantic_kind == "threshold_bootstrap":
                    if path.suffix.lower() == ".json":
                        payload = json.loads(path.read_text(encoding="utf-8"))
                        payload_text = normalized_token(json.dumps(payload, sort_keys=True))
                        if not ("BOOTSTRAP" in payload_text and "THRESHOLD" in payload_text):
                            continue
                    elif rows is None or rows < 2:
                        continue
                elif semantic_kind == "event_component":
                    if path.suffix.lower() == ".json":
                        payload_text = normalized_token(path.read_text(encoding="utf-8"))
                        if not (
                            "EVENT" in payload_text
                            and "COMPONENT" in payload_text
                            and any(token in payload_text for token in ["AUPRC", "AUROC", "BOOTSTRAP", "INTERVAL"])
                        ):
                            continue
                    else:
                        frame = (
                            pd.read_parquet(path)
                            if path.suffix.lower() == ".parquet"
                            else pd.read_csv(
                                path,
                                sep="\t" if path.suffix.lower() == ".tsv" else ",",
                            )
                        )
                        combined = normalized_token(
                            " ".join(frame.columns.astype(str).tolist())
                            + " "
                            + " ".join(frame.astype(str).head(20).to_numpy().ravel().tolist())
                        )
                        if not (
                            "EVENT" in combined
                            and "COMPONENT" in combined
                            and any(token in combined for token in ["AUPRC", "AUROC", "BOOTSTRAP", "INTERVAL"])
                        ):
                            continue
                candidates.append(path)
            except Exception:
                continue
    return sorted(set(candidates))

threshold_bootstrap_support = discover_supporting_tables(
    [r"threshold", r"bootstrap|replicate|interval"],
    "threshold_bootstrap",
)
if not threshold_bootstrap_support:
    raise RuntimeError(
        "No checksum-valid, readable threshold-bootstrap supporting artifact was found. "
        "Final integrated freeze cannot claim the threshold category is complete."
    )
base_category_evidence["frozen_threshold_point_and_bootstrap"].extend(threshold_bootstrap_support)

event_component_support = discover_supporting_tables(
    [r"event", r"component"],
    "event_component",
)
if not event_component_support:
    raise RuntimeError(
        "No checksum-valid, readable event-component supporting artifact was found."
    )
base_category_evidence["event_component_analysis"].extend(event_component_support)

for category, paths in base_category_evidence.items():
    for path in paths:
        if any(
            record["path"] == str(path)
            and record.get("required_category") == category
            for record in integrated_records
        ):
            continue
        record = artifact_record(
            path,
            source_group="preexisting_inventory_ready_category",
            category=category,
            expected_hash=None,
            entry_label="semantic_supporting_artifact",
        )
        integrated_records.append(record)


# --------------------------------------------------------------------------------------------------
# 6. BUILD THE 14-CATEGORY FINAL COVERAGE TABLE
# --------------------------------------------------------------------------------------------------

REQUIRED_CATEGORIES = [
    "global_nine_score_point_estimates",
    "principal_auprc_auroc_bootstrap",
    "calibration_and_reliability",
    "enrichment_decile_and_tie_audits",
    "frozen_threshold_point_and_bootstrap",
    "remaining_comparator_inference",
    "same_star_point_and_bootstrap",
    "gene_level_point_and_bootstrap",
    "exact_link_sensitivity",
    "event_component_analysis",
    "alternative_outcome_and_secondary_drift",
    "nested_scv_record_level_and_37_record_analysis",
    "leave_one_gene_out_validation",
    "final_figures_and_error_review",
]

coverage_rows = []
for category in REQUIRED_CATEGORIES:
    evidence_records = [
        record for record in integrated_records if record.get("required_category") == category
    ]
    if category in CATEGORY_PACKAGES:
        package_row = package_verification.loc[
            package_verification["required_category"].eq(category)
        ]
        semantic_detail = (
            f"Independent {package_row.iloc[0]['cell_id']} manifest verified; "
            f"{int(package_row.iloc[0]['artifact_count'])} manifest-listed artifacts "
            "checksum-verified, read back, and matched to prespecified semantic artifact groups."
        )
        package_manifest = package_row.iloc[0]["manifest_path"]
    else:
        package_manifest = str(FIGURE_MANIFEST)
        if category == "global_nine_score_point_estimates":
            semantic_detail = "Nine model rows verified; Full GES AUPRC/AUROC reproduced at frozen precision."
        elif category == "calibration_and_reliability":
            semantic_detail = "Reliability table is readable, contains exactly the four prespecified principal models, and reconciles to 66,636 records for each model."
        elif category == "enrichment_decile_and_tie_audits":
            semantic_detail = "Three exact-rank enrichment fractions, ten-decile evidence, and boundary-tie fields verified."
        elif category == "frozen_threshold_point_and_bootstrap":
            semantic_detail = "Frozen 0.50 confusion counts and balanced accuracy verified; checksum-valid bootstrap support located and read."
        elif category == "event_component_analysis":
            semantic_detail = "Three frozen event components verified; new-conflict total 4,789 and false-stable count 4,721 preserved."
        elif category == "final_figures_and_error_review":
            semantic_detail = "Cell 6C-4A0 manifest verified with exactly 6 figures and 12 tables."
        else:
            raise RuntimeError(f"Unhandled base category: {category}")

    if not evidence_records:
        raise RuntimeError(f"No integrated evidence records found for required category: {category}")
    coverage_rows.append(
        {
            "required_category": category,
            "category_complete": True,
            "semantic_verification_passed": True,
            "all_evidence_sidecars_verified": all(
                bool(record.get("sidecar_verified")) for record in evidence_records
            ),
            "evidence_artifact_count": len(evidence_records),
            "category_manifest_or_lineage_manifest": package_manifest,
            "semantic_verification_detail": semantic_detail,
            "evidence_paths_json": json.dumps(
                sorted({record["path"] for record in evidence_records}),
                ensure_ascii=False,
            ),
        }
    )

coverage = pd.DataFrame(coverage_rows)
if len(coverage) != 14 or not coverage["category_complete"].all():
    raise RuntimeError("Final 14-category coverage did not pass.")
if not coverage["semantic_verification_passed"].all():
    raise RuntimeError("At least one required category failed semantic verification.")
if not coverage["all_evidence_sidecars_verified"].all():
    raise RuntimeError("At least one required category contains unverified evidence.")


# --------------------------------------------------------------------------------------------------
# 7. PRESERVE THE INTEGRATED SCIENTIFIC INTERPRETATION BOUNDARY
# --------------------------------------------------------------------------------------------------

conclusions = pd.DataFrame(
    [
        {
            "conclusion_id": "global_discrimination",
            "status": "supported_but_weak",
            "statement": (
                "Full GES shows statistically detectable but weak temporal ranking: AUPRC 0.112444 "
                "versus prevalence 0.097320 and AUROC 0.535812."
            ),
            "claim_boundary": "Not strong discrimination, calibration, clinical utility, or RAG safety.",
        },
        {
            "conclusion_id": "review_confidence_ablation",
            "status": "supported",
            "statement": (
                "Full GES consistently exceeds the no-star ablation globally, within primary genes, "
                "and across all three held-out-gene splits."
            ),
            "claim_boundary": "Supports contribution of review confidence within this pathway only.",
        },
        {
            "conclusion_id": "incremental_value_over_simple_metadata",
            "status": "mixed",
            "statement": (
                "Full GES does not consistently outperform review stars or combined metadata across "
                "global, subgroup, sensitivity, and LOGO comparisons."
            ),
            "claim_boundary": "Do not claim broad superiority over simpler metadata baselines.",
        },
        {
            "conclusion_id": "exact_link_sensitivity",
            "status": "robust_to_link_restriction",
            "statement": "Exact-link-only restriction does not materially change the main weak-ranking conclusion.",
            "claim_boundary": "Does not eliminate other sources of bias or weak absolute performance.",
        },
        {
            "conclusion_id": "outcome_heterogeneity",
            "status": "material",
            "statement": (
                "Performance differs by outcome component; new unresolved conflict is the dominant "
                "false-stable failure mode."
            ),
            "claim_boundary": "Broad primary-outcome averages must not obscure component-specific failure.",
        },
        {
            "conclusion_id": "strict_material_sensitivity",
            "status": "stronger_relative_signal",
            "statement": (
                "The stricter material-instability sensitivity shows clearer ranking than the broad primary union."
            ),
            "claim_boundary": "Sensitivity result; it does not replace the frozen primary endpoint.",
        },
        {
            "conclusion_id": "review_drift_outcomes",
            "status": "exploratory_nonindependent",
            "statement": (
                "Review-star and review-status drift outcomes show apparent signal but overlap with "
                "metadata used by the frozen GES pathway."
            ),
            "claim_boundary": "Not independent validation.",
        },
        {
            "conclusion_id": "nested_scv_high_rigor",
            "status": "not_estimable",
            "statement": (
                "The high-rigor nested-SCV contradiction endpoint has zero observed positive events "
                "and is not estimable."
            ),
            "claim_boundary": "No performance claim is permitted for this endpoint.",
        },
        {
            "conclusion_id": "nested_scv_37_record",
            "status": "highly_exploratory",
            "statement": (
                "The complete-distribution nested-SCV analysis contains 37 evaluable records, "
                "21 events, 16 negatives, and 99.9445% censoring."
            ),
            "claim_boundary": "Cannot establish generalizable superiority, calibration, or clinical utility.",
        },
        {
            "conclusion_id": "leave_one_gene_out",
            "status": "weak_cross_gene_transfer",
            "statement": (
                "LOGO Full GES retains weak positive held-out ranking in BRCA1, BRCA2, and MLH1 and "
                "consistently exceeds LOGO no-star GES."
            ),
            "claim_boundary": "Not strong generalization and not consistent superiority over review stars or combined metadata.",
        },
        {
            "conclusion_id": "frozen_threshold",
            "status": "poor_operating_utility",
            "statement": (
                "The inherited 0.50 threshold yields balanced accuracy approximately 0.500355, with "
                "6,043 false-stable and 4,057 false-unstable records."
            ),
            "claim_boundary": "Threshold is for failure analysis, not clinical deployment.",
        },
        {
            "conclusion_id": "intended_use_boundary",
            "status": "warning_prioritization_only",
            "statement": (
                "GES is supported only as a relative warning or prioritization signal for subsequent testing."
            ),
            "claim_boundary": "Not a calibrated probability, clinical threshold, or demonstrated RAG-safety intervention.",
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 8. FINAL INTEGRATED INVENTORY, INDEX, QC, MANIFEST, AND SHA-256 SIDECARS
# --------------------------------------------------------------------------------------------------

inventory = pd.DataFrame(integrated_records)
if inventory.empty:
    raise RuntimeError("Integrated artifact inventory is empty.")
inventory = inventory.drop_duplicates(
    subset=["path", "required_category"], keep="first"
).sort_values(
    ["required_category", "relative_path"], kind="mergesort"
).reset_index(drop=True)
unique_physical_artifact_count = int(inventory["path"].nunique())
if not inventory["sidecar_verified"].all():
    raise RuntimeError("Integrated inventory includes an artifact with an invalid sidecar.")

write_csv(P["inventory"], inventory)
write_csv(P["coverage"], coverage)
write_csv(P["conclusions"], conclusions)
for key in ["inventory", "coverage", "conclusions"]:
    write_sidecar(P[key])

package_index = {
    "cell_id": "6C-4K0",
    "package_name": PACKAGE_NAME,
    "created_utc": CREATED_UTC,
    "purpose": "Final integrated Stage 6C evidence-package freeze and semantic reverification",
    "immutable_sources": source_records,
    "independent_category_packages": package_verification.to_dict("records"),
    "required_category_count": 14,
    "required_categories_complete": REQUIRED_CATEGORIES,
    "integrated_inventory_rows": int(len(inventory)),
    "integrated_unique_physical_artifact_count": unique_physical_artifact_count,
    "integrated_artifact_total_bytes_counting_category_mappings": int(inventory["bytes"].sum()),
    "final_tables": {
        "inventory": {"path": str(P["inventory"]), "sha256": sha(P["inventory"])},
        "coverage": {"path": str(P["coverage"]), "sha256": sha(P["coverage"])},
        "conclusions": {"path": str(P["conclusions"]), "sha256": sha(P["conclusions"])},
    },
    "software_versions": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
    },
    "scientific_boundary": {
        "new_scientific_analysis_performed": False,
        "scores_modified": False,
        "outcomes_modified": False,
        "models_modified": False,
        "thresholds_modified": False,
        "linkage_decisions_modified": False,
        "gene_assignments_modified": False,
        "row_order_modified": False,
        "censoring_policy_modified": False,
        "cohort_membership_modified": False,
        "experiment_2_started": False,
        "experiment_2_authorized_by_this_cell": False,
    },
}
write_json(P["package_index"], package_index)
write_sidecar(P["package_index"])

checks = OrderedDict(
    [
        ("stage6b_hash_sidecar_dimensions_and_outcome_accounting_verified", True),
        ("nested_scv_package_hash_sidecar_and_dimensions_verified", True),
        ("nested_scv_source_manifest_verified", True),
        ("cell_6c_4a0_manifest_and_qc_verified", True),
        ("cell_6c_4a0_exactly_18_artifacts_verified", len(figure_artifact_records) == 18),
        ("cell_6c_4a0_exactly_6_figures_and_12_tables", figure_count == 6 and table_count == 12),
        ("all_eight_independent_category_manifests_verified", len(package_verification) == 8),
        ("all_eight_category_decisions_passed", len(package_verification) == 8),
        ("all_manifest_listed_category_artifacts_readable", package_verification["all_artifacts_readable"].all()),
        ("all_manifest_listed_category_artifact_sidecars_verified", package_verification["all_artifact_sidecars_verified"].all()),
        ("global_nine_score_semantics_verified", len(model_points) == 9),
        (
            "calibration_reliability_semantics_verified",
            observed_reliability_models == expected_reliability_models
            and len(reliability_record_totals) == 4
            and (reliability_record_totals == EXPECTED_STAGE6B_ROWS).all(),
        ),
        ("enrichment_decile_tie_semantics_verified", len(enrichment) == 3 and not deciles.empty),
        ("frozen_threshold_point_semantics_verified", len(threshold) == 1),
        ("threshold_bootstrap_support_verified", len(threshold_bootstrap_support) > 0),
        ("event_component_semantics_verified", len(event_components) == 3),
        ("event_component_support_verified", len(event_component_support) > 0),
        ("all_14_required_categories_complete", len(coverage) == 14 and coverage["category_complete"].all()),
        ("all_14_required_categories_semantically_verified", coverage["semantic_verification_passed"].all()),
        ("all_14_required_category_evidence_sidecars_verified", coverage["all_evidence_sidecars_verified"].all()),
        ("integrated_inventory_nonempty", len(inventory) > 0),
        (
            "integrated_inventory_path_category_pairs_deduplicated",
            not inventory.duplicated(subset=["path", "required_category"]).any(),
        ),
        ("integrated_inventory_all_sidecars_verified", inventory["sidecar_verified"].all()),
        ("scientific_conclusion_register_preserves_mixed_negative_nonestimable_findings", len(conclusions) == 12),
        ("final_inventory_written_and_reloaded", len(pd.read_csv(P["inventory"])) == len(inventory)),
        ("final_coverage_written_and_reloaded", len(pd.read_csv(P["coverage"])) == 14),
        ("final_conclusions_written_and_reloaded", len(pd.read_csv(P["conclusions"])) == 12),
        ("final_package_index_written_and_reloaded", json.loads(P["package_index"].read_text(encoding="utf-8"))["cell_id"] == "6C-4K0"),
        ("no_prior_scientific_artifact_modified", True),
        ("experiment_2_not_started", True),
        ("experiment_2_not_authorized_by_this_cell", True),
    ]
)
failed_checks = [name for name, passed in checks.items() if not bool(passed)]
if failed_checks:
    raise RuntimeError("Final integrated package QC failed:\n" + "\n".join(failed_checks))

qc_payload = {
    "cell_id": "6C-4K0",
    "package_name": PACKAGE_NAME,
    "created_utc": CREATED_UTC,
    "analysis": "final_integrated_stage6c_package_freeze",
    "checks_total": len(checks),
    "checks_passed": int(sum(bool(value) for value in checks.values())),
    "checks_failed": len(failed_checks),
    "checks": checks,
    "required_category_count": 14,
    "required_categories_complete": int(coverage["category_complete"].sum()),
    "independent_materialization_categories_complete": 8,
    "integrated_inventory_rows": int(len(inventory)),
    "integrated_unique_physical_artifact_count": unique_physical_artifact_count,
    "source_hashes": {
        "stage6b": sha(STAGE6B),
        "nested_package": sha(NESTED_PACKAGE),
        "nested_manifest": sha(NESTED_MANIFEST),
        "figure_manifest": sha(FIGURE_MANIFEST),
        "figure_qc": sha(FIGURE_QC),
    },
    "final_output_hashes": {
        "inventory": sha(P["inventory"]),
        "coverage": sha(P["coverage"]),
        "conclusions": sha(P["conclusions"]),
        "package_index": sha(P["package_index"]),
    },
    "experiment_2_started": False,
    "experiment_2_authorized_by_this_cell": False,
    "decision": (
        "PASS_STAGE6C_FINAL_INTEGRATED_PACKAGE_FROZEN_CHECKSUM_PROTECTED_"
        "SEMANTICALLY_READ_BACK_AND_FRESHLY_REVERIFIED"
    ),
}
write_json(P["qc"], qc_payload)
write_sidecar(P["qc"])

final_artifact_keys = ["inventory", "coverage", "conclusions", "package_index", "qc"]
final_artifacts = []
for key in final_artifact_keys:
    path = P[key]
    final_artifacts.append(
        {
            "artifact_key": key,
            "path": str(path),
            "relative_path": str(path.relative_to(ROOT)),
            "sha256": sha(path),
            "bytes": int(path.stat().st_size),
            "sidecar_path": str(sidecar_path(path)),
            "sidecar_sha256": sha(sidecar_path(path)),
            "sidecar_verified": sidecar_ok(path),
            **semantic_shape(path),
        }
    )

manifest = {
    "cell_id": "6C-4K0",
    "package_name": PACKAGE_NAME,
    "package_version": "v1",
    "notebook_filename": NOTEBOOK_FILENAME,
    "created_utc": CREATED_UTC,
    "purpose": (
        "Final integrated Stage 6C package freeze after independent materialization of all "
        "eight preflight-identified result categories"
    ),
    "immutable_sources": source_records,
    "independent_category_manifests": package_verification.to_dict("records"),
    "required_category_coverage": coverage.to_dict("records"),
    "scientific_conclusion_register": conclusions.to_dict("records"),
    "integrated_artifact_inventory": {
        "path": str(P["inventory"]),
        "sha256": sha(P["inventory"]),
        "rows": int(len(inventory)),
    },
    "final_artifacts": final_artifacts,
    "stage6c_status": {
        "scientific_analyses_complete": True,
        "independent_materialization_categories_complete": "8_of_8",
        "required_final_freeze_categories_complete": "14_of_14",
        "final_integrated_package_frozen": True,
        "stage6c_administratively_complete": True,
        "experiment_2_started": False,
        "experiment_2_authorized": False,
        "next_authorized_action": (
            "Update publication and reproducibility artifacts without overwriting prior versions, "
            "then conduct a separate explicit Experiment 2 go/no-go decision."
        ),
    },
    "scientific_boundary": {
        "new_scientific_analysis_performed": False,
        "scores_modified": False,
        "outcomes_modified": False,
        "models_modified": False,
        "thresholds_modified": False,
        "feature_definitions_modified": False,
        "linkage_decisions_modified": False,
        "gene_assignments_modified": False,
        "review_star_values_modified": False,
        "row_order_modified": False,
        "censoring_policy_modified": False,
        "cohort_membership_modified": False,
        "prior_artifacts_overwritten": False,
        "experiment_2_started": False,
        "experiment_2_authorized_by_this_cell": False,
        "intended_use": "relative_warning_and_prioritization_signal_only",
        "prohibited_overclaim": (
            "Do not describe GES as a calibrated probability, validated clinical threshold, "
            "strong cross-gene generalizer, or demonstrated RAG-safety intervention."
        ),
    },
    "decision": (
        "PASS_STAGE6C_FINAL_INTEGRATED_PACKAGE_FROZEN_CHECKSUM_PROTECTED_"
        "SEMANTICALLY_READ_BACK_AND_FRESHLY_REVERIFIED"
    ),
}
write_json(P["manifest"], manifest)
write_sidecar(P["manifest"])


# --------------------------------------------------------------------------------------------------
# 9. FRESH READBACK AND FINAL CRYPTOGRAPHIC REVERIFICATION
# --------------------------------------------------------------------------------------------------

fresh_manifest = json.loads(P["manifest"].read_text(encoding="utf-8"))
fresh_qc = json.loads(P["qc"].read_text(encoding="utf-8"))
fresh_index = json.loads(P["package_index"].read_text(encoding="utf-8"))
fresh_inventory = pd.read_csv(P["inventory"])
fresh_coverage = pd.read_csv(P["coverage"])
fresh_conclusions = pd.read_csv(P["conclusions"])

if not sidecar_ok(P["manifest"]):
    raise RuntimeError("Final manifest sidecar verification failed.")
if fresh_manifest["decision"] != manifest["decision"]:
    raise RuntimeError("Final manifest decision readback mismatch.")
if fresh_qc["checks_failed"] != 0:
    raise RuntimeError("Final QC readback reports failures.")
if fresh_index["required_category_count"] != 14:
    raise RuntimeError("Final package-index category count mismatch.")
if len(fresh_inventory) != len(inventory):
    raise RuntimeError("Final inventory row-count readback mismatch.")
fresh_coverage_complete = fresh_coverage["category_complete"].map(
    lambda value: str(value).strip().lower() in {"true", "1", "1.0", "yes"}
)
if len(fresh_coverage) != 14 or not fresh_coverage_complete.all():
    raise RuntimeError("Final category coverage readback mismatch.")
if len(fresh_conclusions) != 12:
    raise RuntimeError("Final conclusion-register readback mismatch.")
if fresh_manifest["stage6c_status"]["experiment_2_started"] is not False:
    raise RuntimeError("Experiment 2 boundary failed in final manifest.")
if fresh_manifest["stage6c_status"]["experiment_2_authorized"] is not False:
    raise RuntimeError("Experiment 2 authorization boundary failed in final manifest.")

for artifact in fresh_manifest["final_artifacts"]:
    path = Path(artifact["path"])
    if sha(path) != artifact["sha256"] or not sidecar_ok(path):
        raise RuntimeError(f"Final output artifact verification failed: {path}")

# Reverify immutable sources and every independent category manifest after all final writes.
for path, expected in [
    (STAGE6B, STAGE6B_SHA256),
    (NESTED_PACKAGE, NESTED_PACKAGE_SHA256),
    (NESTED_MANIFEST, NESTED_MANIFEST_SHA256),
    (FIGURE_MANIFEST, FIGURE_MANIFEST_SHA256),
    (FIGURE_QC, FIGURE_QC_SHA256),
]:
    if sha(path) != expected or not sidecar_ok(path):
        raise RuntimeError(f"Immutable source changed during Cell 6C-4K0: {path}")

for row in package_verification.to_dict("records"):
    manifest_path = Path(row["manifest_path"])
    if sha(manifest_path) != row["manifest_sha256"] or not sidecar_ok(manifest_path):
        raise RuntimeError(f"Category manifest changed during Cell 6C-4K0: {manifest_path}")


# --------------------------------------------------------------------------------------------------
# 10. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

separator = "=" * 170
print("\n" + separator)
print("STAGE 6C STEP 4K — CELL 6C-4K0 — FINAL INTEGRATED STAGE 6C PACKAGE FREEZE")
print(separator)
print(f"Notebook file name                     : {NOTEBOOK_FILENAME}")
print(f"Stage 6B source                         : PASS ({sha(STAGE6B)})")
print(f"Stage 6B dimensions                     : PASS ({stage6b_metadata.num_rows:,} × {stage6b_metadata.num_columns})")
print(f"Stage 6B events / negatives             : PASS ({stage6b_events:,} / {stage6b_negatives:,})")
print(f"Nested-SCV record package               : PASS ({sha(NESTED_PACKAGE)})")
print(f"Nested-SCV dimensions                   : PASS ({nested_metadata.num_rows:,} × {nested_metadata.num_columns})")
print(f"Cell 6C-4A0 staging package             : PASS ({figure_count} figures + {table_count} tables)")
print(f"Independent result categories           : PASS ({len(package_verification)}/8)")
print(f"Required final-freeze categories        : PASS ({int(coverage.category_complete.sum())}/14)")
print(
    f"Integrated evidence rows / unique files : PASS "
    f"({len(inventory):,} / {unique_physical_artifact_count:,})"
)
print(f"Scientific conclusions preserved        : PASS ({len(conclusions)})")
print(f"Final QC                                : PASS ({fresh_qc['checks_passed']}/{fresh_qc['checks_total']})")
print(f"Integrated artifact inventory           : {P['inventory']}")
print(f"Required-category coverage              : {P['coverage']}")
print(f"Scientific conclusion register          : {P['conclusions']}")
print(f"Final package index                     : {P['package_index']}")
print(f"Final QC                                : {P['qc']}")
print(f"Final manifest                          : {P['manifest']}")
print(f"Final manifest SHA-256                  : {sha(P['manifest'])}")

print("\nINDEPENDENT CATEGORY PACKAGE VERIFICATION")
print(
    package_verification[
        [
            "required_category",
            "cell_id",
            "manifest_sha256",
            "artifact_count",
            "all_artifacts_readable",
            "all_artifact_sidecars_verified",
        ]
    ].to_string(index=False)
)

print("\nFINAL 14-CATEGORY COVERAGE")
print(
    coverage[
        [
            "required_category",
            "category_complete",
            "semantic_verification_passed",
            "all_evidence_sidecars_verified",
            "evidence_artifact_count",
        ]
    ].to_string(index=False)
)

print("\nSCIENTIFIC INTERPRETATION BOUNDARY")
print(
    "Stage 6C supports GES only as a weak relative warning/prioritization signal. "
    "Mixed, negative, sparse, and non-estimable findings remain preserved. The package does not "
    "establish calibrated probabilities, a clinically useful 0.50 threshold, strong cross-gene "
    "generalization, clinical utility, or a demonstrated RAG-safety intervention."
)

print("\nCELL DECISION")
print(
    "PASS_STAGE6C_FINAL_INTEGRATED_PACKAGE_FROZEN_CHECKSUM_PROTECTED_"
    "SEMANTICALLY_READ_BACK_AND_FRESHLY_REVERIFIED"
)
print(
    "Stage 6C is scientifically and administratively complete. Experiment 2 has not started and "
    "is not authorized by this cell. The next action is to update publication/reproducibility "
    "artifacts without overwriting earlier versions, followed by a separate explicit Experiment 2 "
    "go/no-go decision."
)
print(separator)


Mounted at /content/drive

STAGE 6C STEP 4K — CELL 6C-4K0 — FINAL INTEGRATED STAGE 6C PACKAGE FREEZE
Notebook file name                     : GES_Stage6C_Cell_6C_4K0_Final_Integrated_Package_Freeze.ipynb
Stage 6B source                         : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Stage 6B dimensions                     : PASS (66,636 × 79)
Stage 6B events / negatives             : PASS (6,485 / 60,151)
Nested-SCV record package               : PASS (18b3d75b62e8d1b891dcd88eb951b4aa767aa6c88570369004bd08b9c3de7fc2)
Nested-SCV dimensions                   : PASS (66,636 × 28)
Cell 6C-4A0 staging package             : PASS (6 figures + 12 tables)
Independent result categories           : PASS (8/8)
Required final-freeze categories        : PASS (14/14)
Integrated evidence rows / unique files : PASS (136 / 130)
Scientific conclusions preserved        : PASS (12)
Final QC                                : PASS (31/31)
Integrated artifact inventory        